# 🎬 Task 2: Movie Genre Classification
## TF-IDF & Machine Learning – CodSoft AI/ML Internship

**Google Colab Edition**: This notebook is optimized for Google Colab. It will mount your Google Drive to save the trained model so you can use it later.

## 💾 Step 1: Mount Google Drive & Setup Directories

In [ ]:
from google.colab import drive
import os

# Mount Google Drive to save the model and outputs permanently
drive.mount('/content/drive')

# Define paths on Google Drive for saving models
SAVE_DIR = '/content/drive/MyDrive/CodSoft_Task2'
MODELS_DIR = os.path.join(SAVE_DIR, 'models')
os.makedirs(MODELS_DIR, exist_ok=True)
print(f'✅ Models will be saved to: {MODELS_DIR}')

## 🔑 Step 2: Download Dataset using Kaggle API Token

In [ ]:
import os
from getpass import getpass

# Ask for the Kaggle API Token (starts with KGAT_...)
print('🔑 Please enter your Kaggle API Token:')
token = getpass('Token: ')
os.environ['KAGGLE_API_TOKEN'] = token

DATA_DIR = '/content/dataset'
TRAIN_PATH = os.path.join(DATA_DIR, 'Genre Classification Dataset/train_data.txt')

if not os.path.exists(TRAIN_PATH):
    print('📥 Downloading dataset from Kaggle...')
    !pip install kaggle -q
    os.makedirs(DATA_DIR, exist_ok=True)
    !kaggle datasets download -d hijest/genre-classification-dataset-imdb -p {DATA_DIR} --unzip
    print('✅ Dataset downloaded and unzipped!')
else:
    print('✅ Dataset already exists.')

## 📦 Step 3: Install Dependencies & Import Libraries

In [ ]:
!pip install pandas numpy scikit-learn nltk matplotlib seaborn tqdm joblib gradio -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import joblib
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from tqdm import tqdm

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

nltk.download('stopwords')
nltk.download('punkt')
print('✅ All libraries imported successfully!')

## 📊 Step 4: Load & Explore the Dataset

In [ ]:
def load_data(filepath):
    # The dataset uses ' ::: ' as a separator
    try:
        df = pd.read_csv(filepath, sep=' ::: ', engine='python', names=['ID', 'Title', 'Genre', 'Description'])
        return df
    except Exception as e:
        print(f"Error loading data: {e}")
        return None

print("Loading Training Data...")
train_df = load_data(os.path.join(DATA_DIR, 'Genre Classification Dataset/train_data.txt'))
print(f"Training Data Shape: {train_df.shape}")
print(train_df.head())

print("\nLoading Test Data...")
test_df = load_data(os.path.join(DATA_DIR, 'Genre Classification Dataset/test_data_solution.txt'))
if test_df is None:  # Fallback if solution file is named differently or missing
    test_df = load_data(os.path.join(DATA_DIR, 'Genre Classification Dataset/test_data.txt'))
print(f"Test Data Shape: {test_df.shape}")

## 📈 Step 5: Exploratory Data Analysis (EDA)

In [ ]:
plt.figure(figsize=(14, 7))
sns.countplot(y='Genre', data=train_df, order=train_df['Genre'].value_counts().index, palette='viridis')
plt.title('Distribution of Movie Genres in Training Data', fontsize=16)
plt.xlabel('Number of Movies', fontsize=12)
plt.ylabel('Genre', fontsize=12)
plt.tight_layout()
plt.show()

print(f"Total unique genres: {train_df['Genre'].nunique()}")

## 🧹 Step 6: Text Preprocessing

In [ ]:
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def preprocess_text(text):
    # Convert to lowercase
    text = text.lower()
    # Remove punctuation and special characters
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    # Tokenize, remove stopwords, and apply stemming
    words = text.split()
    words = [stemmer.stem(word) for word in words if word not in stop_words]
    return ' '.join(words)

print("Applying text preprocessing to training data (this may take a minute)...")
tqdm.pandas()
train_df['Clean_Description'] = train_df['Description'].progress_apply(preprocess_text)

print("\nApplying text preprocessing to test data...")
test_df['Clean_Description'] = test_df['Description'].progress_apply(preprocess_text)

print("\nBefore & After Preprocessing (Sample):")
print("Original:", train_df['Description'].iloc[0])
print("Cleaned :", train_df['Clean_Description'].iloc[0])

## 🔢 Step 7: Feature Extraction (TF-IDF)

In [ ]:
# Initialize TF-IDF Vectorizer
# Using max_features to limit memory usage and speed up training
tfidf_vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))

print("Fitting TF-IDF Vectorizer on training data...")
X_train = tfidf_vectorizer.fit_transform(train_df['Clean_Description'])
y_train = train_df['Genre']

print("Transforming test data...")
X_test = tfidf_vectorizer.transform(test_df['Clean_Description'])
y_test = test_df['Genre'] if 'Genre' in test_df.columns else None

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")

## 🤖 Step 8: Model Training (Logistic Regression & Naive Bayes)

In [ ]:
# Model 1: Logistic Regression (Usually performs well on text classification)
print("Training Logistic Regression Model...")
lr_model = LogisticRegression(max_iter=1000, n_jobs=-1, class_weight='balanced', C=3.0)
lr_model.fit(X_train, y_train)
print("✅ Logistic Regression Training Complete!")

# Model 2: Multinomial Naive Bayes
print("\nTraining Naive Bayes Model...")
nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)
print("✅ Naive Bayes Training Complete!")

## 📈 Step 9: Model Evaluation

In [ ]:
if y_test is not None:
    print("--- Logistic Regression Evaluation ---")
    y_pred_lr = lr_model.predict(X_test)
    print(f"Accuracy: {accuracy_score(y_test, y_pred_lr):.4f}")
    print(classification_report(y_test, y_pred_lr, zero_division=0))
    
    print("\n--- Naive Bayes Evaluation ---")
    y_pred_nb = nb_model.predict(X_test)
    print(f"Accuracy: {accuracy_score(y_test, y_pred_nb):.4f}")
else:
    print("Test labels not available for evaluation.")

## 💾 Step 10: Save Models to Google Drive

In [ ]:
lr_model_path = os.path.join(MODELS_DIR, 'logistic_regression_model.pkl')
tfidf_path = os.path.join(MODELS_DIR, 'tfidf_vectorizer.pkl')

joblib.dump(lr_model, lr_model_path)
joblib.dump(tfidf_vectorizer, tfidf_path)

print(f"✅ Model saved to: {lr_model_path}")
print(f"✅ Vectorizer saved to: {tfidf_path}")

## 🎮 Step 11: Interactive Movie Genre Predictor

In [ ]:
def predict_genre(plot_summary):
    # 1. Clean the text
    cleaned_text = preprocess_text(plot_summary)
    # 2. Vectorize
    vectorized_text = tfidf_vectorizer.transform([cleaned_text])
    # 3. Predict
    prediction = lr_model.predict(vectorized_text)[0]
    return prediction

# Test with some custom plots
custom_plots = [
    "A group of astronauts travel through a wormhole in search of a new habitable planet for humanity.",
    "A detective investigates a series of gruesome murders in a gloomy city, chasing a serial killer who uses the seven deadly sins as his motive.",
    "A young boy discovers he is a wizard and attends a magical school where he must fight an evil dark lord.",
    "Two people meet on a train and slowly fall in love over the course of a single night in Vienna."
]

print("--- Custom Plot Predictions ---")
for plot in custom_plots:
    genre = predict_genre(plot)
    print(f"\nPlot: {plot}")
    print(f"Predicted Genre: -> ** {genre.upper()} ** <-")


## 🌐 Step 12: Interactive Web UI using Gradio (Bonus!)

In [ ]:
import gradio as gr

def ui_predict_genre(plot_summary):
    cleaned_text = preprocess_text(plot_summary)
    vectorized_text = tfidf_vectorizer.transform([cleaned_text])
    prediction = lr_model.predict(vectorized_text)[0]
    
    # Get probabilities to show confidence
    probs = lr_model.predict_proba(vectorized_text)[0]
    max_prob = probs.max() * 100
    
    return f"🎬 {prediction.upper()} (Confidence: {max_prob:.2f}%)"

examples_list = [
    ["A young boy discovers he possesses incredible magical abilities and is whisked away to a secret school for wizards, where he must battle a dark lord who threatens the entire realm."],
    ["In a post-apocalyptic wasteland, a drifter is captured by a tyrannical cult but manages to escape with a group of female captives, leading to an intense, explosive desert car chase."],
    ["Two strangers meet on a train in Europe and decide to spend one magical, romantic evening walking through the streets of Vienna before they part ways forever at sunrise."],
    ["When a ruthless serial killer begins targeting victims based on the seven deadly sins, two detectives must race against time to stop the gruesome murders before they become the next targets."]
]

interface2 = gr.Interface(
    fn=ui_predict_genre,
    inputs=gr.Textbox(lines=5, placeholder="Enter a detailed movie plot summary here... (The more details, the better the prediction!)"),
    outputs=gr.Textbox(label="Predicted Genre"),
    title="🎬 Movie Genre Classifier",
    description="Predicts the genre of a movie based on its plot using an advanced TF-IDF and Logistic Regression pipeline.\n\n**Tip:** The model is trained on full movie summaries. Extremely short sentences (like 'a boy got magic') don't give the AI enough context. Try writing a descriptive paragraph!",
    examples=examples_list
)

interface2.launch(share=True)